# Gurobi: stručný recept na praktický test
**Obecný postup → syntaxe → samostatná řešená zadání**

První část je obecný tahák. Názvy `n`, `k`, `w`, `B` apod. jsou v každé ukázce vysvětlené a zastupují tvoje vlastní data. Ukázky v textu jsou **vzory, ne jeden společný program**.

Na konci jsou tři řešené úlohy. Každá má nejprve vlastní zadání a potom kompletní spustitelnou buňku s daty. K jejich pochopení nepotřebuješ žádný jiný dokument.

### Vždy stejné pořadí
1. **Vstup:** načti data a sjednoť indexy na `0 … n-1`.
2. **Proměnné:** napiš si, co znamená každá jejich hodnota.
3. **Omezení:** každou podmínku zadání přelož zvlášť.
4. **Cíl:** nastav výraz a `MAXIMIZE` nebo `MINIMIZE`.
5. **Řešení:** zavolej `optimize()` a zkontroluj stav.
6. **Výstup:** přečti `.X` a zapiš přesný formát zadání.

Jednotný styl: **`import gurobipy as g`**, **`model`**, **`g.quicksum`**, **vnější `for` + `addConstr`**. Samotnou matematickou formulaci musíš zvolit podle úlohy.


## 1. Proměnné: `addVar` a `addVars`
```python
import gurobipy as g
model = g.Model()

# Jedna promenna ano/ne.
y = model.addVar(vtype=g.GRB.BINARY, name="y")

# n nezapornych celociselnych promennych: x[0], ..., x[n-1].
x = model.addVars(n, vtype=g.GRB.INTEGER, lb=0, name="x")

# Tabulka n x k promennych ano/ne: pristup a[i, j].
a = model.addVars(n, k, vtype=g.GRB.BINARY, name="a")
```

| Zápis | Význam |
|---|---|
| `addVar` | Jedna proměnná; používáš třeba `y`. |
| `addVars` | Kolekce proměnných; používáš `x[i]` nebo `a[i, j]`. |
| `g.GRB.INTEGER` | Celé číslo. |
| `g.GRB.BINARY` | Jen 0 nebo 1. |
| `g.GRB.CONTINUOUS` | Spojitá hodnota. |
| `lb`, `ub` | Dolní a horní mez. Výchozí dolní mez je 0. |

Pro záporné i kladné hodnoty dej `lb=-g.GRB.INFINITY`. Výchozí horní mez je nekonečná, u `BINARY` je doména 0/1. `name` je nepovinný název pro ladění.

V čistém ILP jsou rozhodovací proměnné celočíselné, vstupní koeficienty mohou být desetinné. Kombinace celočíselných a spojitých proměnných je MILP. Proměnné vytváříš v modelu; čísla načtená ze vstupu zůstávají běžná data.


## 2. Omezení a součty: jeden způsob zápisu
**`addConstr` přidá jedno omezení.** Můžeš ho volat opakovaně v cyklu. `addConstrs` existuje, ale v tomto návodu ho nepotřebuješ.

Nechť `x[i]` jsou počty, `u[i]` jejich známé horní meze, `w[i]` známé spotřeby a `B` celková kapacita:
```python
# n samostatnych omezeni.
for i in range(n):
    model.addConstr(x[i] <= u[i])

# Jedno omezeni se souctem n clenu.
model.addConstr(g.quicksum(w[i] * x[i] for i in range(n)) <= B)
```

**Vnější `for` = pro každý index samostatné omezení. `for` uvnitř `quicksum` = členy jednoho součtu.**

Pro tabulku binárních proměnných `a[i, j]` má každý řádek `i` nejvýše jednu jedničku:
```python
for i in range(n):
    model.addConstr(g.quicksum(a[i, j] for j in range(k)) <= 1)
```
Pro každý sloupec `j` nejvýše jednu jedničku:
```python
for j in range(k):
    model.addConstr(g.quicksum(a[i, j] for i in range(n)) <= 1)
```
Celou tabulku sečteš `g.quicksum(a[i, j] for i in range(n) for j in range(k))`.

| Požadavek | Porovnání |
|---|---|
| Nejvýše K | `soucet <= K` |
| Alespoň K | `soucet >= K` |
| Právě K | `soucet == K` |

Vyber správný směr. Samotné vytvoření `quicksum(...)` ještě žádné omezení do modelu nepřidá.


## 3. Logické podmínky a Big-M
Následující vzory jsou nezávislé. `a`, `b`, `z`, `y` jsou jednotlivé binární proměnné.

```python
model.addConstr(a + b <= 1)   # Nesmime vybrat oba.
model.addConstr(a + b >= 1)   # Musime vybrat alespon jednoho.
model.addConstr(a <= b)       # Jestlize a=1, pak b=1.
```
**`z = 1` právě tehdy, když `a = b = 1`:**
```python
model.addConstr(z <= a)
model.addConstr(z <= b)
model.addConstr(z >= a + b - 1)
```

**Aktivace nezáporného počtu `q` pomocí `y`:** znáš platnou horní mez `U` pro `q`.
```python
model.addConstr(q <= U * y)
```
Při `y=0` musí být `q=0`. Při `y=1` smí být `q` od 0 do `U`. Pro celočíselné `q` a přesnou ekvivalenci „zapnuto právě když q > 0“ přidej `q >= y`.

**Obecná implikace:** `expr` je lineární výraz, `rhs` známá mez.
```python
model.addConstr((y == 1) >> (expr <= rhs))
```
To je indikátorové omezení: pravá strana musí platit, pokud je `y=1`. Alternativa pomocí Big-M:
```python
model.addConstr(expr <= rhs + big_m * (1 - y))
```
Použij jeden způsob. `big_m` musí být alespoň největší možné `expr - rhs` při vypnuté podmínce; vezmi nejmenší rozumnou prokázanou mez. **Větší platné M nemusí změnit celočíselná řešení, ale oslabuje LP relaxaci a může zhoršit numeriku. Nekonečno do této role nepatří.**

**NEBO mezi dvěma podmínkami:** `P`, `H` jsou lineární výrazy a `U` známá mez; `choose` je binární.
```python
model.addConstr((choose == 0) >> (P >= U))
model.addConstr((choose == 1) >> (H >= U))
```
Alespoň jedna podmínka musí platit; obě současně jsou povolené. Toto není Pythonové `or`.


## 4. Cíl, spuštění a čtení výsledku
`c[i]` je známý koeficient, `x[i]` rozhodovací proměnná. Součet můžeš uložit do běžného názvu `objective`; tím nevzniká nová rozhodovací proměnná.
```python
objective = g.quicksum(c[i] * x[i] for i in range(n))
model.setObjective(objective, g.GRB.MAXIMIZE)  # Nebo MINIMIZE.
model.optimize()

# Jen pri nejednoznacnem stavu zkusime jeho rozliseni.
if model.Status == g.GRB.INF_OR_UNBD:
    model.Params.DualReductions = 0
    model.optimize()

if model.Status == g.GRB.OPTIMAL:
    print("Cil:", model.ObjVal)
    for i in range(n):
        print(i, x[i].X)
elif model.Status == g.GRB.INFEASIBLE:
    print("Zadani nema pripustne reseni.")
else:
    raise RuntimeError(f"Stav solveru: {model.Status}")
```

- **`x[i]`** je objekt proměnné, **`x[i].X`** nalezená hodnota.
- U binární proměnné testuj **`.X > 0.5`**. Celočíselný počet převáděj přes **`round(.X)`**.
- **`model.ObjVal`** je hodnota cíle. Zaokrouhluj ji jen tehdy, má-li být podle zadání celočíselná.
- **`OPTIMAL`** potvrzuje optimum v tolerancích řešiče, **`INFEASIBLE`** prokázanou nesplnitelnost.
- **`TIME_LIMIT` neznamená nesplnitelnost.** `SolCount > 0` dovoluje přečíst dostupné řešení, ale sám nepotvrzuje optimum.


## 5. Načítání a výstup: drž se jednoho stylu
Pro tuto ukázku má vstup tři řádky: samotné číslo `n`, seznam čísel `values` a dvojici čísel `A B`.
```python
with open(input_path, "r") as f:
    n = int(f.readline())
    values = [int(v) for v in f.readline().split()]
    A, B = [int(v) for v in f.readline().split()]
```
`readline()` přečte další řádek, `split()` rozdělí text podle bílých znaků, `int` převede část na číslo. Samotné číslo a seznam s jedním číslem nejsou totéž. U `int(f.readline())` není `.strip()` nutné. Desetinná vstupní data načítej přes `float`.

V notebooku si nastav `input_path = "input.txt"`. Ve skriptu spouštěném jako `python3 main.py input.txt output.txt` použij:
```python
#!/usr/bin/env python3
import sys
input_path = sys.argv[1]
output_path = sys.argv[2]
```
Shebang patří na úplně první řádek skriptu, pokud ho odevzdávací systém vyžaduje. V notebooku `sys.argv` obvykle obsahuje argumenty kernelu.

Pokud zadání chce jedno číslo `value` a pak seznam čísel `answer`:
```python
with open(output_path, "w") as f:
    f.write(str(value) + "\n")
    f.write(" ".join(str(v) for v in answer) + "\n")
```
Formát vždy uprav podle zadání: někdy je každá hodnota na novém řádku, někdy se cíl vůbec nevypisuje. `"w"` přepisuje soubor. Indexy začínající na 1 převeď při vstupu na 0 a při výstupu zpátky, pokud to zadání požaduje.


## 6. Kontrola před odevzdáním
| Častá chyba | Správný návyk |
|---|---|
| `if x[i] == 1` při tvorbě modelu | Použij matematické nebo indikátorové omezení. `if` nad známými vstupními daty je v pořádku. |
| `a[i][j]` u Gurobi kolekce | Piš `a[i, j]`. Běžná matice dat může používat `cost[i][j]`. |
| `=` místo porovnání | V omezení piš `==`, `<=` nebo `>=`. |
| `0 <= x <= 10` | Dej `lb=0, ub=10`, nebo dvě samostatná omezení. |
| Pythonové `and`/`or` mezi omezeními | Použij lineární logické vzory nebo indikátory. |
| Součin dvou proměnných v ILP | Není lineární; pro binární AND použij tři omezení. Číslo krát proměnná je v pořádku. |
| Jiný výstup než v ukázce | Může být také optimální; kontroluj přípustnost a hodnotu cíle. |

Ověř domény, všechny podmínky zadání, směr cíle, indexy a výstup. Model můžeš prohlédnout přes `model.write("debug.lp")`. Pro prokázanou nesplnitelnost lze zavolat `model.computeIIS()` a `model.write("conflict.ilp")`; získáš konfliktní podmnožinu, ne nutně jedinou ani nejmenší.

Pokud chybí `gurobipy`, zvol Python/kernel s instalovaným balíčkem. Chyba licence je jiný problém. Konkrétní školní prostředí a způsob odevzdání se řídí pokyny kurzu.


# Řešené úlohy
Tato část je volitelná. Každá následující kódová buňka obsahuje **vlastní import, data, model a výpis výsledku**. Lze ji spustit samostatně v prostředí s funkčním Gurobi.

## A. Barvení domů
Máme šest domů s indexy 0–5 a tři barvy 0–2. Každý dům musí mít právě jednu barvu. Dvojice sousedů jsou `(0,1), (2,0), (3,2), (1,3), (3,4), (4,5)`; sousedé nesmějí mít stejnou barvu.

| Barva | Cena za jeden dům | Zásoba v počtu domů |
|---|---:|---:|
| 0 | 3 | 3 |
| 1 | 1 | 2 |
| 2 | 8 | 6 |

**Cíl:** minimalizovat cenu při dodržení zásob. `x[i,j]=1` znamená, že dům `i` dostal barvu `j`. Vypíšeme cenu a seznam barev domů v pořadí 0–5. Očekávaná optimální cena je **19**; přiřazení nemusí být jediné.


In [ ]:
import gurobipy as g

n, k = 6, 3
cost, volume = [3, 1, 8], [3, 2, 6]
edges = [(0, 1), (2, 0), (3, 2), (1, 3), (3, 4), (4, 5)]

model = g.Model("barveni")
model.Params.OutputFlag = 0
model.Params.DualReductions = 0
x = model.addVars(n, k, vtype=g.GRB.BINARY, name="x")

for i in range(n):
    model.addConstr(g.quicksum(x[i, j] for j in range(k)) == 1)
for j in range(k):
    model.addConstr(g.quicksum(x[i, j] for i in range(n)) <= volume[j])
for u, v in edges:
    for j in range(k):
        model.addConstr(x[u, j] + x[v, j] <= 1)

model.setObjective(
    g.quicksum(cost[j] * x[i, j] for i in range(n) for j in range(k)),
    g.GRB.MINIMIZE
)
model.optimize()

if model.Status == g.GRB.OPTIMAL:
    answer = []
    for i in range(n):
        for j in range(k):
            if x[i, j].X > 0.5:
                answer.append(j)
                break
    print("Cena:", round(model.ObjVal), "Barvy:", answer)
elif model.Status == g.GRB.INFEASIBLE:
    print("Nelze obarvit.")
else:
    raise RuntimeError(model.Status)


## B. Jezdci na šachovnici
Na šachovnici 8 × 8 stojí pevné věže na **b2** a **d5**. Umísti co nejvíce jezdců tak, aby se žádní dva neohrožovali a žádný nestál na políčku ohroženém věží. Pro tuto úlohu věž zakazuje **celý svůj řádek i sloupec**, včetně vlastního políčka; blokování jinými figurami neuvažujeme.

Jezdec ohrožuje pole vzdálené o 1 řádek a 2 sloupce nebo o 2 řádky a 1 sloupec. Řádky i sloupce v kódu číslujeme od 0. `x[r,c]=1` znamená obsazené pole. Výstupem je počet a seznam šachových souřadnic. Očekávané optimum je **20**.

Použijeme jen čtyři směry s kladnou změnou řádku. Každou neorientovanou konfliktní dvojici tak zahrneme právě jednou.


In [ ]:
import gurobipy as g

rooks = [(1, 1), (4, 3)]  # b2, d5: (radek, sloupec)
model = g.Model("jezdci")
model.Params.OutputFlag = 0
x = model.addVars(8, 8, vtype=g.GRB.BINARY, name="x")

for rr, cc in rooks:
    for r in range(8):
        model.addConstr(x[r, cc] == 0)
    for c in range(8):
        model.addConstr(x[rr, c] == 0)

moves = [(1, 2), (1, -2), (2, 1), (2, -1)]
for r in range(8):
    for c in range(8):
        for dr, dc in moves:
            rr, cc = r + dr, c + dc
            if 0 <= rr < 8 and 0 <= cc < 8:
                model.addConstr(x[r, c] + x[rr, cc] <= 1)

model.setObjective(
    g.quicksum(x[r, c] for r in range(8) for c in range(8)),
    g.GRB.MAXIMIZE
)
model.optimize()

if model.Status != g.GRB.OPTIMAL:
    raise RuntimeError(model.Status)
answer = []
for r in range(8):
    for c in range(8):
        if x[r, c].X > 0.5:
            answer.append(chr(ord("a") + c) + str(r + 1))
print("Pocet:", len(answer), "Policka:", answer)


## C. Horníci: přiřazení, přátelství a NEBO
Máme čtyři horníky 0–3 a tři pracovní místa 0–2. Velikosti míst jsou `[0, 1, 2]`.

| Horník | Cech | Velikost | Spokojenost | Produkce |
|---|---:|---:|---:|---:|
| 0 | 0 | 0 | 2 | 6 |
| 1 | 1 | 1 | 3 | 4 |
| 2 | 2 | 2 | 4 | 5 |
| 3 | 0 | 0 | 1 | 2 |

Podmínky:
1. Každý horník dostane nejvýše jedno místo, každé místo nejvýše jednoho horníka. Nikdo nemusí být vybraný povinně.
2. Horník nesmí být větší než přidělené místo.
3. Počet vybraných horníků z každého cechu musí být nejvýše součet počtů ze zbývajících dvou cechů.
4. Dvojice přátel jsou `(0,1)` a `(1,2)`. Za každou dvojici, jejíž oba členové jsou vybraní, přibude **5 bodů spokojenosti**, bez ohledu na jejich místa.
5. Celková produkce **nebo** celková spokojenost včetně bonusů musí být alespoň **18**. Mohou platit obě podmínky.

**Maximalizujeme produkci + spokojenost včetně bonusů.**

`x[m,l]` znamená přiřazení, `y[m]` přítomnost horníka, `z[f]` přítomnost obou přátel dvojice `f`. Rovnost `sum_l x[m,l] == y[m]` současně zajistí nejvýše jedno místo a propojí přítomnost s přiřazením. Binární `choose` zvolí, kterou z podmínek NEBO musíme splnit.

Vypíšeme cíl a místa horníků v pořadí 0–3; `-1` znamená nepřiřazený. Očekávaný cíl je **34**, vybráni jsou horníci 0, 1 a 2.


In [ ]:
import gurobipy as g

M, L = 4, 3
guild, size = [0, 1, 2, 0], [0, 1, 2, 0]
dim, happy, prod = [0, 1, 2], [2, 3, 4, 1], [6, 4, 5, 2]
friends, U = [(0, 1), (1, 2)], 18

model = g.Model("hornici")
model.Params.OutputFlag = 0
model.Params.DualReductions = 0
x = model.addVars(M, L, vtype=g.GRB.BINARY, name="x")
y = model.addVars(M, vtype=g.GRB.BINARY, name="y")
z = model.addVars(len(friends), vtype=g.GRB.BINARY, name="z")
choose = model.addVar(vtype=g.GRB.BINARY, name="choose")

for m in range(M):
    model.addConstr(g.quicksum(x[m, l] for l in range(L)) == y[m])
for l in range(L):
    model.addConstr(g.quicksum(x[m, l] for m in range(M)) <= 1)
for m in range(M):
    for l in range(L):
        if size[m] > dim[l]:
            model.addConstr(x[m, l] == 0)
for c in range(3):
    own = g.quicksum(y[m] for m in range(M) if guild[m] == c)
    other = g.quicksum(y[m] for m in range(M) if guild[m] != c)
    model.addConstr(own <= other)
for f, (a, b) in enumerate(friends):
    model.addConstr(z[f] <= y[a])
    model.addConstr(z[f] <= y[b])
    model.addConstr(z[f] >= y[a] + y[b] - 1)

P = g.quicksum(prod[m] * y[m] for m in range(M))
H = (g.quicksum(happy[m] * y[m] for m in range(M))
     + 5 * g.quicksum(z[f] for f in range(len(friends))))
model.addConstr((choose == 0) >> (P >= U))
model.addConstr((choose == 1) >> (H >= U))
model.setObjective(P + H, g.GRB.MAXIMIZE)
model.optimize()

if model.Status == g.GRB.OPTIMAL:
    assignment = [-1] * M
    for m in range(M):
        for l in range(L):
            if x[m, l].X > 0.5:
                assignment[m] = l
    print("Cil:", round(model.ObjVal), "Mista:", assignment)
elif model.Status == g.GRB.INFEASIBLE:
    print("Neexistuje pripustne prirazeni.")
else:
    raise RuntimeError(model.Status)


### Poznámka ke spouštění a zdroje
V řešeních `OutputFlag = 0` potlačuje log. U modelů, které mohou být nesplnitelné, `DualReductions = 0` pomáhá rozlišit nesplnitelnost od neomezenosti. Tyto parametry nemění formulaci úlohy.

Řešené úlohy jsou samostatně popsané adaptace zadání *Painting the neighborhood*, *Knights Placement* a *Mine of Kuttenberg*. Příklady vypisují výsledky do notebooku; formát souborů při odevzdání se řídí konkrétním zadáním.

Oficiální reference: [Model a jeho metody](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html) · [quicksum](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/func_global.html) · [omezení](https://docs.gurobi.com/projects/optimizer/en/current/concepts/modeling/constraints.html) · [stavy řešiče](https://docs.gurobi.com/projects/optimizer/en/current/reference/numericcodes/statuscodes.html).
